# Movie ABT Enrichment with TMDB and IMDB Data

## Overview
This notebook enriches your Analytics Base Table (ABT) with comprehensive movie data from:
- **TMDB** (The Movie Database)
- **IMDB** (Internet Movie Database)

## What it does:
1. Reads your ABT CSV/Excel file
2. For each movie, fetches data from TMDB using tmdbId
3. For each movie, fetches data from IMDB using imdbId
4. Adds all enriched attributes to the ABT
5. Saves enhanced ABT as Excel file

## TMDB Fields Retrieved:
- id, title, original_title, overview, tagline, status
- release_date, runtime, budget, revenue, popularity
- genres, original_language, spoken_languages

## IMDB Fields Retrieved:
- id, Title, Genres, Rating, Cast, Overview
- Runtime, Release year, Poster URL, Keywords

In [ ]:
# Install required packages
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai beautifulsoup4 requests --quiet

print("✓ Packages installed")

In [1]:
# Import libraries
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, Optional
import re

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from langchain_community.document_loaders import WebBaseLoader
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

print("✓ Imports successful")

✓ Imports successful


In [2]:
# Load environment variables
load_dotenv()
load_dotenv('../.env')

# Initialize LLM for content extraction
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✓ LLM initialized")

✓ LLM initialized


## Configuration

In [3]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION - UPDATE THESE PATHS
# ═══════════════════════════════════════════════════════════

# Input file (CSV or Excel)
INPUT_FILE = "../data/outputs/analytics_base_table_with_genres_list.csv" 

# Output file (will be saved in same directory as input)
OUTPUT_FILE = "analytics_base_table_ENRICHED.xlsx"

# How many movies to process (set to None for all)
# MAX_MOVIES = None  # Process all movies
MAX_MOVIES = 10  # Process first 10 for testing

# Rate limiting (seconds between requests)
SLEEP_BETWEEN_REQUESTS = 1.5  # Be respectful to servers

print("✓ Configuration loaded")
print(f"  Input file: {INPUT_FILE}")
print(f"  Output file: {OUTPUT_FILE}")
print(f"  Max movies: {MAX_MOVIES if MAX_MOVIES else 'All'}")

✓ Configuration loaded
  Input file: ../data/outputs/analytics_base_table_with_genres_list.csv
  Output file: analytics_base_table_ENRICHED.xlsx
  Max movies: 10


## Data Fetching Functions

In [4]:
def fetch_tmdb_data(tmdb_id: str) -> Optional[Dict]:
    """
    Fetch movie data from TMDB.
    
    URL format: https://www.themoviedb.org/movie/{tmdb_id}
    
    Returns dictionary with:
    - tmdb_title, tmdb_original_title, tmdb_overview, tmdb_tagline
    - tmdb_status, tmdb_release_date, tmdb_runtime, tmdb_budget
    - tmdb_revenue, tmdb_popularity, tmdb_genres
    - tmdb_original_language, tmdb_spoken_languages
    """
    if pd.isna(tmdb_id) or str(tmdb_id).strip() == '':
        return None
    
    try:
        url = f"https://www.themoviedb.org/movie/{tmdb_id}"
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Use LLM to extract structured data
        extraction_prompt = f"""
Extract movie information from this TMDB page and return as JSON.

Extract these fields (use null if not found):
- title: Movie title
- original_title: Original title if different
- overview: Plot synopsis/description
- tagline: Movie tagline
- status: Release status (Released, Planned, etc.)
- release_date: Release date (YYYY-MM-DD format)
- runtime: Runtime in minutes (number only)
- budget: Budget in dollars (number only)
- revenue: Revenue in dollars (number only)
- popularity: Popularity score (number)
- genres: List of genre names
- original_language: Original language code (e.g., 'en')
- spoken_languages: List of spoken language names

HTML Content:
{response.text[:8000]}

Return ONLY valid JSON, no other text.
"""
        
        result = llm.invoke([HumanMessage(content=extraction_prompt)])
        
        # Parse JSON from response
        json_str = result.content.strip()
        if json_str.startswith('```json'):
            json_str = json_str.split('```json')[1].split('```')[0].strip()
        elif json_str.startswith('```'):
            json_str = json_str.split('```')[1].split('```')[0].strip()
        
        data = json.loads(json_str)
        
        # Prefix all fields with tmdb_
        return {f"tmdb_{k}": v for k, v in data.items()}
        
    except Exception as e:
        print(f"  ⚠ TMDB fetch failed for {tmdb_id}: {e}")
        return None


def fetch_imdb_data(imdb_id: str) -> Optional[Dict]:
    """
    Fetch movie data from IMDB.
    
    URL format: https://www.imdb.com/title/{imdb_id}/
    
    Returns dictionary with:
    - imdb_id, imdb_title, imdb_genres, imdb_rating
    - imdb_cast, imdb_overview, imdb_runtime
    - imdb_release_year, imdb_poster_url, imdb_keywords
    """
    if pd.isna(imdb_id) or str(imdb_id).strip() == '':
        return None
    
    try:
        url = f"https://www.imdb.com/title/{imdb_id}/"
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Use LLM to extract structured data
        extraction_prompt = f"""
Extract movie information from this IMDB page and return as JSON.

Extract these fields (use null if not found):
- id: IMDB ID (e.g., 'tt0113041')
- title: Movie title
- genres: List of genre names
- rating: IMDB rating (number, e.g., 7.5)
- cast: List of main cast member names (top 5-10)
- overview: Plot synopsis
- runtime: Runtime in minutes (number only)
- release_year: Release year (number, e.g., 1995)
- poster_url: URL to movie poster image
- keywords: List of keywords/tags

HTML Content:
{response.text[:8000]}

Return ONLY valid JSON, no other text.
"""
        
        result = llm.invoke([HumanMessage(content=extraction_prompt)])
        
        # Parse JSON from response
        json_str = result.content.strip()
        if json_str.startswith('```json'):
            json_str = json_str.split('```json')[1].split('```')[0].strip()
        elif json_str.startswith('```'):
            json_str = json_str.split('```')[1].split('```')[0].strip()
        
        data = json.loads(json_str)
        
        # Prefix all fields with imdb_
        return {f"imdb_{k}": v for k, v in data.items()}
        
    except Exception as e:
        print(f"  ⚠ IMDB fetch failed for {imdb_id}: {e}")
        return None


print("✓ Fetch functions defined")

✓ Fetch functions defined


## Load ABT Data

In [5]:
# Read input file
input_path = Path(INPUT_FILE)

if not input_path.exists():
    # Try in data folder
    input_path = Path("../data") / INPUT_FILE
    if not input_path.exists():
        raise FileNotFoundError(f"Could not find {INPUT_FILE}")

print(f"Reading: {input_path}")

# Read based on file extension
if input_path.suffix == '.csv':
    df = pd.read_csv(input_path)
else:
    df = pd.read_excel(input_path)

print(f"✓ Loaded {len(df)} movies")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(3)

Reading: ../data/outputs/analytics_base_table_with_genres_list.csv
✓ Loaded 9742 movies

Columns: ['Unnamed: 0', 'movieId', 'title', 'genres', 'avg rating ', 'no of viewer rating ', 'viewers tags', 'imdbId', 'tmdbId', 'Genres_List']

First few rows:


,Unnamed: 0,movieId,title,genres,avg rating,no of viewer rating,viewers tags,imdbId,tmdbId,Genres_List
0,0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.9,215.0,pixar|pixar|fun,114709,862.0,"['Adventure', 'Animation', 'Children', 'Comedy..."
1,1,2,Jumanji (1995),Adventure|Children|Fantasy,3.4,110.0,fantasy|magic board game|Robin Williams|game,113497,8844.0,"['Adventure', 'Children', 'Fantasy']"
2,2,3,Grumpier Old Men (1995),Comedy|Romance,3.3,52.0,moldy|old,113228,15602.0,"['Comedy', 'Romance']"


In [6]:
# Limit to MAX_MOVIES if specified
if MAX_MOVIES:
    df = df.head(MAX_MOVIES)
    print(f"✓ Limited to {len(df)} movies for processing")

✓ Limited to 10 movies for processing


## Enrich Data from TMDB and IMDB

In [7]:
# Initialize new columns for TMDB data
tmdb_columns = [
    'tmdb_title', 'tmdb_original_title', 'tmdb_overview', 'tmdb_tagline',
    'tmdb_status', 'tmdb_release_date', 'tmdb_runtime', 'tmdb_budget',
    'tmdb_revenue', 'tmdb_popularity', 'tmdb_genres',
    'tmdb_original_language', 'tmdb_spoken_languages'
]

# Initialize new columns for IMDB data
imdb_columns = [
    'imdb_id', 'imdb_title', 'imdb_genres', 'imdb_rating',
    'imdb_cast', 'imdb_overview', 'imdb_runtime',
    'imdb_release_year', 'imdb_poster_url', 'imdb_keywords'
]

# Add columns to dataframe
for col in tmdb_columns + imdb_columns:
    if col not in df.columns:
        df[col] = None

print(f"✓ Added {len(tmdb_columns + imdb_columns)} new columns")

✓ Added 23 new columns


In [8]:
# Process each movie
print("="*70)
print("ENRICHING MOVIES WITH TMDB AND IMDB DATA")
print("="*70)
print(f"Processing {len(df)} movies...\n")

start_time = time.time()
success_count = {'tmdb': 0, 'imdb': 0}
fail_count = {'tmdb': 0, 'imdb': 0}

for idx, row in df.iterrows():
    movie_title = row.get('title', 'Unknown')
    print(f"\n[{idx+1}/{len(df)}] {movie_title}")
    
    # Fetch TMDB data
    tmdb_id = row.get('tmdbId')
    if pd.notna(tmdb_id):
        print(f"  Fetching TMDB (ID: {tmdb_id})...")
        tmdb_data = fetch_tmdb_data(str(int(tmdb_id)))
        
        if tmdb_data:
            for key, value in tmdb_data.items():
                df.at[idx, key] = value
            print(f"  ✓ TMDB data retrieved")
            success_count['tmdb'] += 1
        else:
            fail_count['tmdb'] += 1
        
        time.sleep(SLEEP_BETWEEN_REQUESTS)
    
    # Fetch IMDB data
    imdb_id = row.get('imdbId')
    if pd.notna(imdb_id):
        # Format IMDB ID (add 'tt' prefix if missing)
        imdb_id_str = str(int(imdb_id))
        if not imdb_id_str.startswith('tt'):
            imdb_id_str = f"tt{imdb_id_str.zfill(7)}"
        
        print(f"  Fetching IMDB (ID: {imdb_id_str})...")
        imdb_data = fetch_imdb_data(imdb_id_str)
        
        if imdb_data:
            for key, value in imdb_data.items():
                df.at[idx, key] = value
            print(f"  ✓ IMDB data retrieved")
            success_count['imdb'] += 1
        else:
            fail_count['imdb'] += 1
        
        time.sleep(SLEEP_BETWEEN_REQUESTS)
    
    # Progress update every 10 movies
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df) - idx - 1) / rate if rate > 0 else 0
        print(f"\n  Progress: {idx+1}/{len(df)} | Rate: {rate:.1f} movies/sec | ETA: {remaining/60:.1f} min")

elapsed_time = time.time() - start_time

print("\n" + "="*70)
print("ENRICHMENT COMPLETE")
print("="*70)
print(f"Total time: {elapsed_time/60:.1f} minutes")
print(f"\nTMDB: {success_count['tmdb']} successful, {fail_count['tmdb']} failed")
print(f"IMDB: {success_count['imdb']} successful, {fail_count['imdb']} failed")

ENRICHING MOVIES WITH TMDB AND IMDB DATA
Processing 10 movies...


[1/10] Toy Story (1995)
  Fetching TMDB (ID: 862.0)...
  ✓ TMDB data retrieved
  Fetching IMDB (ID: tt0114709)...
  ✓ IMDB data retrieved

[2/10] Jumanji (1995)
  Fetching TMDB (ID: 8844.0)...
  ✓ TMDB data retrieved
  Fetching IMDB (ID: tt0113497)...
  ✓ IMDB data retrieved

[3/10] Grumpier Old Men (1995)
  Fetching TMDB (ID: 15602.0)...
  ✓ TMDB data retrieved
  Fetching IMDB (ID: tt0113228)...
  ✓ IMDB data retrieved

[4/10] Waiting to Exhale (1995)
  Fetching TMDB (ID: 31357.0)...
  ✓ TMDB data retrieved
  Fetching IMDB (ID: tt0114885)...
  ✓ IMDB data retrieved

[5/10] Father of the Bride Part II (1995)
  Fetching TMDB (ID: 11862.0)...
  ✓ TMDB data retrieved
  Fetching IMDB (ID: tt0113041)...
  ✓ IMDB data retrieved

[6/10] Heat (1995)
  Fetching TMDB (ID: 949.0)...
  ✓ TMDB data retrieved
  Fetching IMDB (ID: tt0113277)...
  ✓ IMDB data retrieved

[7/10] Sabrina (1995)
  Fetching TMDB (ID: 11860.0)...
  ✓ TMDB da

## Preview Enriched Data

In [9]:
# Show sample of enriched data
print("Sample of enriched data:")
print("="*70)

# Show first movie with all new fields
sample_movie = df.iloc[0]
print(f"\nMovie: {sample_movie['title']}")
print("\nTMDB Fields:")
for col in tmdb_columns:
    if col in df.columns:
        print(f"  {col}: {sample_movie[col]}")

print("\nIMDB Fields:")
for col in imdb_columns:
    if col in df.columns:
        print(f"  {col}: {sample_movie[col]}")

Sample of enriched data:

Movie: Toy Story (1995)

TMDB Fields:
  tmdb_title: Toy Story
  tmdb_original_title: Toy Story
  tmdb_overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.
  tmdb_tagline: None
  tmdb_status: None
  tmdb_release_date: 1995-11-22
  tmdb_runtime: None
  tmdb_budget: None
  tmdb_revenue: None
  tmdb_popularity: None
  tmdb_genres: []
  tmdb_original_language: en
  tmdb_spoken_languages: []

IMDB Fields:
  imdb_id: tt0114709
  imdb_title: Toy Story
  imdb_genres: ['Animation', 'Adventure', 'Comedy']
  imdb_rating: 8.3
  imdb_cast: ['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim Varney', 'Annie Potts']
  imdb_overview: A cowboy doll is profoundly jealous when a new spaceman action figure supplants him as the to

In [ ]:
# Show statistics on data completeness
print("\nData Completeness:")
print("="*70)

all_new_cols = tmdb_columns + imdb_columns
for col in all_new_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        pct = (non_null / len(df)) * 100
        print(f"{col:30} {non_null:4}/{len(df):4} ({pct:5.1f}%)")

## Save Enriched ABT

In [11]:
# Determine output path (same directory as input)
output_path = input_path.parent / OUTPUT_FILE

# Define all_new_cols for summary statistics
all_new_cols = tmdb_columns + imdb_columns

print(f"Saving enriched ABT to: {output_path}")

# Save as Excel
df.to_excel(output_path, index=False, engine='openpyxl')

print(f"✓ Saved {len(df)} enriched movies")
print(f"✓ Total columns: {len(df.columns)}")
print(f"  - Original columns: {len(df.columns) - len(all_new_cols)}")
print(f"  - New TMDB columns: {len(tmdb_columns)}")
print(f"  - New IMDB columns: {len(imdb_columns)}")
print(f"\n✓ File saved: {output_path}")

Saving enriched ABT to: ../data/outputs/analytics_base_table_ENRICHED.xlsx
✓ Saved 10 enriched movies
✓ Total columns: 33
  - Original columns: 10
  - New TMDB columns: 13
  - New IMDB columns: 10

✓ File saved: ../data/outputs/analytics_base_table_ENRICHED.xlsx


In [12]:
# Also save a CSV version for backup
csv_output = output_path.with_suffix('.csv')
df.to_csv(csv_output, index=False)
print(f"✓ CSV backup saved: {csv_output}")

✓ CSV backup saved: ../data/outputs/analytics_base_table_ENRICHED.csv


## Summary Report

In [ ]:
print("="*70)
print("ENRICHMENT SUMMARY REPORT")
print("="*70)
print(f"\nInput file:  {input_path}")
print(f"Output file: {output_path}")
print(f"\nMovies processed: {len(df)}")
print(f"Processing time:  {elapsed_time/60:.1f} minutes")
print(f"\nSuccess rates:")
print(f"  TMDB: {success_count['tmdb']}/{len(df)} ({success_count['tmdb']/len(df)*100:.1f}%)")
print(f"  IMDB: {success_count['imdb']}/{len(df)} ({success_count['imdb']/len(df)*100:.1f}%)")
print(f"\nNew columns added: {len(all_new_cols)}")
print(f"  TMDB columns: {len(tmdb_columns)}")
print(f"  IMDB columns: {len(imdb_columns)}")
print(f"\nTotal columns in enriched file: {len(df.columns)}")
print("="*70)

## Done! 🎉

Your enriched ABT file has been saved with comprehensive movie data from both TMDB and IMDB.

### Next Steps:
1. Open the Excel file to review the enriched data
2. Use the new fields for enhanced movie recommendations
3. Build better search and filtering capabilities
4. Create richer movie profiles for your chatbot

### Files Created:
- Excel: Contains all enriched data with formatting
- CSV: Backup copy for easy importing